# Daily Feature Completion: Wind Direction & NA Handling

**Purpose:** Apply the final cleaning steps to the merged daily feature table
before aggregation to monthly resolution:

1. **Add wind direction category** — bin `wind_from_direction` (degrees) into 8 compass octants
2. **Fix temperature inversions** — where `max_air_temperature < min_air_temperature`, set max = min
3. **Check relative humidity** — flag rows where max RH < min RH (logged, not corrected)
4. **Verify grid consistency** — confirm same (lat, lon) set across dates
5. **Fill NAs** — zero-fill selected columns; drop rows missing core weather variables

**Two execution modes:**
- **Phase A — Sample inspection** (first 200k rows): fast QA before committing to full run
- **Phase B — Full batch processing**: memory-safe row-group iteration via PyArrow

> **Note:** PDSI and OpenStreetMap features are intentionally *not* present in this file.
> They are joined at monthly resolution in `04_01_Agg_PDSI_OpenStreetMap_to_Monthly.ipynb`
> to avoid processing the full ~127M-row daily table.

**Input:** `Clean_Data/Feature_Data/Weather_Population_LAI_Merged.parquet`

**Output:** `Clean_Data/Feature_Data/Daily_Features_Completed.parquet`

## 0. Configuration

Centralized path configuration — **edit this cell only**.

In [1]:
import os

# ====================== EDIT THESE PATHS ======================
PROJECT_ROOT = r"E:\zcao\CA_Wildfire"

INPUT_PATH  = os.path.join(PROJECT_ROOT, "Clean_Data", "Feature_Data",
                           "Weather_Population_LAI_Merged.parquet")
OUTPUT_PATH = os.path.join(PROJECT_ROOT, "Clean_Data", "Feature_Data",
                           "Daily_Features_Completed.parquet")
LOG_DIR     = os.path.join(PROJECT_ROOT, "Logs", "Feature_Engineering")
LOG_FILE    = "wind_direction_fill_na_log.txt"

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

# Wind direction bin edges (N wraps around 0°)
WIND_DIRECTION_RANGES = {
    'N' : (337.5, 22.5),  'NE': (22.5,  67.5),
    'E' : (67.5,  112.5), 'SE': (112.5, 157.5),
    'S' : (157.5, 202.5), 'SW': (202.5, 247.5),
    'W' : (247.5, 292.5), 'NW': (292.5, 337.5),
}
WIND_DIRECTION_ORDER = ['N','NE','E','SE','S','SW','W','NW']

# Columns to fill with 0 (PDSI is NOT listed — kept as NaN)
FILL_ZERO_COLS = [
    'precipitation_amount', 'LAI', 'population_density', 'SWE',
]
# Columns where rows with NaN are dropped entirely
DROP_NA_COLS = [
    'dead_fuel_moisture_1000hr', 'dead_fuel_moisture_100hr',
    'max_relative_humidity', 'min_relative_humidity',
    'specific_humidity', 'surface_downwelling_shortwave_flux_in_air',
    'wind_speed', 'wind_from_direction', 'wind_direction_category',
    'max_air_temperature', 'min_air_temperature',
]

print(f"Input  : {INPUT_PATH}")
print(f"Output : {OUTPUT_PATH}")
print(f"Log    : {os.path.join(LOG_DIR, LOG_FILE)}")
for label, path in [("INPUT_PATH", INPUT_PATH)]:
    print(f"  [{'OK' if os.path.exists(path) else 'MISSING'}] {label}")

Input  : E:\zcao\CA_Wildfire\Clean_Data\Feature_Data\Weather_Population_LAI_Merged.parquet
Output : E:\zcao\CA_Wildfire\Clean_Data\Feature_Data\Daily_Features_Completed.parquet
Log    : E:\zcao\CA_Wildfire\Logs\Feature_Engineering\wind_direction_fill_na_log.txt
  [OK] INPUT_PATH


## 1. Environment Setup

In [2]:
import sys, gc, warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import pyproj
import pyarrow as pa
import pyarrow.parquet as pq
from datetime import datetime

pd.set_option('display.max_colwidth', None)
gc.collect()

print(f"Python : {sys.version.split('|')[0].strip()}")
print(f"pandas : {pd.__version__}")
print(f"numpy  : {np.__version__}")
print(f"pyproj : {pyproj.__version__}")

Python : 3.9.13 (main, Aug 25 2022, 23:51:50) [MSC v.1916 64 bit (AMD64)]
pandas : 2.2.2
numpy  : 1.24.4
pyproj : 3.6.1


---

# Phase A: Sample Inspection

Load the first 200k rows to quickly validate column types, missing rates,
and logic before running the full batch in Phase B.

## 2. Load Sample (200k rows)

In [3]:
SAMPLE_ROWS = 200_000

pf = pq.ParquetFile(INPUT_PATH)
print(f"Total rows      : {pf.metadata.num_rows:,}")
print(f"Row groups      : {pf.metadata.num_row_groups}")
print(f"Loading sample  : {SAMPLE_ROWS:,} rows")

sample = next(pf.iter_batches(batch_size=SAMPLE_ROWS)).to_pandas()
print(f"Sample shape    : {sample.shape}")
print(f"\nColumn dtypes:")
print(sample.dtypes.to_string())

Total rows      : 127,478,960
Row groups      : 122
Loading sample  : 200,000 rows
Sample shape    : (200000, 18)

Column dtypes:
day                                          datetime64[ns]
lat                                                 float64
lon                                                 float64
SWE                                                 float32
year                                                  int32
dead_fuel_moisture_1000hr                           float64
dead_fuel_moisture_100hr                            float64
max_air_temperature                                 float64
max_relative_humidity                               float64
min_air_temperature                                 float64
min_relative_humidity                               float64
precipitation_amount                                float64
specific_humidity                                   float64
surface_downwelling_shortwave_flux_in_air           float64
wind_from_direction           

## 3. Sample QA: Missing Rates & Date Range

In [4]:
print(f"Date range : {sample['day'].min().date()} → {sample['day'].max().date()}")
print()
missing = sample.isnull().mean().mul(100).sort_values(ascending=False)
missing = missing[missing > 0]
print("Missing rate (%) — non-zero columns only:")
print(missing.to_string())

Date range : 1994-01-01 → 1994-01-16

Missing rate (%) — non-zero columns only:
LAI                                          5.1305
SWE                                          1.7045
min_relative_humidity                        0.3605
wind_speed                                   0.3605
wind_from_direction                          0.3605
surface_downwelling_shortwave_flux_in_air    0.3605
specific_humidity                            0.3605
precipitation_amount                         0.3605
min_air_temperature                          0.3605
max_relative_humidity                        0.3605
max_air_temperature                          0.3605
dead_fuel_moisture_100hr                     0.3605
dead_fuel_moisture_1000hr                    0.3605
population_density                           0.0750


## 4. Sample QA: Wind Direction Binning

In [5]:
# Vectorized np.select for wind direction categories
wd = sample['wind_from_direction'].to_numpy()
conditions = [
    (wd >= 337.5) | (wd < 22.5),
    (wd >= 22.5)  & (wd < 67.5),
    (wd >= 67.5)  & (wd < 112.5),
    (wd >= 112.5) & (wd < 157.5),
    (wd >= 157.5) & (wd < 202.5),
    (wd >= 202.5) & (wd < 247.5),
    (wd >= 247.5) & (wd < 292.5),
    (wd >= 292.5) & (wd < 337.5),
]
sample['wind_direction_category'] = np.select(conditions, WIND_DIRECTION_ORDER, default=np.nan)
sample['wind_direction_category'] = pd.Categorical(
    sample['wind_direction_category'],
    categories=WIND_DIRECTION_ORDER, ordered=True
)

# Verify bins: min/max degree per category should match expected ranges
bin_check = sample.groupby('wind_direction_category').agg(
    min_deg=('wind_from_direction','min'),
    max_deg=('wind_from_direction','max'),
    count=('wind_from_direction','count')
).reset_index()
print(bin_check.to_string(index=False))

wind_direction_category  min_deg  max_deg  count
                      N      0.0    359.0  39126
                     NE     23.0     67.0  35377
                      E     68.0    112.0  17918
                     SE    113.0    157.0  16400
                      S    158.0    202.0  19621
                     SW    203.0    247.0  22128
                      W    248.0    292.0  17796
                     NW    293.0    337.0  30913


## 5. Sample QA: Temperature & Humidity Inversions

In [6]:
bad_temp = (sample['max_air_temperature'] < sample['min_air_temperature']).sum()
bad_hum  = (sample['max_relative_humidity'] < sample['min_relative_humidity']).sum()
print(f"Rows with max_air_temp  < min_air_temp  : {bad_temp:,}")
print(f"Rows with max_rel_humid < min_rel_humid : {bad_hum:,}")

Rows with max_air_temp  < min_air_temp  : 0
Rows with max_rel_humid < min_rel_humid : 0


## 6. Sample QA: Grid Consistency (1994 vs 2006)

In [7]:
grid_1994 = sample[sample['day']=='1994-01-01'][['lat','lon']].drop_duplicates()
grid_2006 = sample[sample['day']=='2006-01-01'][['lat','lon']].drop_duplicates()
print(f"Grid cells 1994-01-01 : {len(grid_1994):,}")
print(f"Grid cells 2006-01-01 : {len(grid_2006):,}")
# (low counts expected — sample may not contain both dates)
del sample; gc.collect()

Grid cells 1994-01-01 : 13,048
Grid cells 2006-01-01 : 0


0

---

# Phase B: Full Batch Processing

Process the entire dataset in row-group batches via PyArrow to avoid
loading all ~127M rows into RAM at once. Each batch applies:

1. Wind direction binning
2. Temperature inversion fix
3. Zero-fill selected columns
4. Drop rows missing core weather variables

Results are streamed to a new Parquet file using a `ParquetWriter`.

## 7. Run Batch Cleaning

In [8]:
import pyarrow as pa
import pyarrow.parquet as pq

BATCH_SIZE = 250_000

log_messages = []
log_messages.append(f"Task: Wind direction + NA cleaning (batch mode)")
log_messages.append(f"Started : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
log_messages.append(f"Input   : {INPUT_PATH}")
log_messages.append(f"Output  : {OUTPUT_PATH}")
log_messages.append(f"Batch   : {BATCH_SIZE:,} rows")

pf = pq.ParquetFile(INPUT_PATH)
total_in = total_out = bad_temp_rows = bad_hum_rows = 0
missing_counts = None
latlon_1994 = latlon_2006 = set()
writer = None

for i, batch in enumerate(pf.iter_batches(batch_size=BATCH_SIZE), start=1):
    df = batch.to_pandas()
    total_in += len(df)

    # Track pre-clean missingness
    bm = df.isnull().sum()
    missing_counts = bm if missing_counts is None else missing_counts.add(bm, fill_value=0)

    # Wind direction
    wd = df['wind_from_direction'].to_numpy()
    conds = [
        (wd>=337.5)|(wd<22.5), (wd>=22.5)&(wd<67.5),
        (wd>=67.5)&(wd<112.5), (wd>=112.5)&(wd<157.5),
        (wd>=157.5)&(wd<202.5),(wd>=202.5)&(wd<247.5),
        (wd>=247.5)&(wd<292.5),(wd>=292.5)&(wd<337.5),
    ]
    df['wind_direction_category'] = np.select(conds, WIND_DIRECTION_ORDER, default=np.nan)
    df['wind_direction_category'] = pd.Categorical(
        df['wind_direction_category'], categories=WIND_DIRECTION_ORDER, ordered=True)

    # Temperature inversion fix
    tmask = (df['max_air_temperature'] < df['min_air_temperature']).fillna(False)
    bad_temp_rows += tmask.sum()
    df.loc[tmask, 'max_air_temperature'] = df.loc[tmask, 'min_air_temperature']

    # Humidity check (logged only, not corrected)
    hmask = (df['max_relative_humidity'] < df['min_relative_humidity']).fillna(False)
    bad_hum_rows += hmask.sum()

    # Grid check
    ts = pd.to_datetime(df['day'], errors='coerce')
    if (ts == pd.Timestamp('1994-01-01')).any():
        latlon_1994 = latlon_1994 | set(map(tuple,
            df.loc[ts=='1994-01-01',['lat','lon']].drop_duplicates().to_numpy()))
    if (ts == pd.Timestamp('2006-01-01')).any():
        latlon_2006 = latlon_2006 | set(map(tuple,
            df.loc[ts=='2006-01-01',['lat','lon']].drop_duplicates().to_numpy()))

    # Fill NAs with 0
    for col in FILL_ZERO_COLS:
        if col in df.columns: df[col] = df[col].fillna(0)

    # Drop rows missing core weather variables
    drop_subset = [c for c in DROP_NA_COLS if c in df.columns]
    df = df.dropna(subset=drop_subset)
    total_out += len(df)

    table = pa.Table.from_pandas(df, preserve_index=False)
    if writer is None:
        writer = pq.ParquetWriter(OUTPUT_PATH, table.schema, compression='snappy')
    writer.write_table(table)

    if i % 50 == 0 or i == 1:
        print(f"Batch {i:3d} | in: {len(batch):>9,} | out: {len(df):>9,} | running out: {total_out:>12,}")

    del df, table, batch; gc.collect()

if writer: writer.close()

print(f"\nBatch cleaning done.")
print(f"Total input rows  : {total_in:,}")
print(f"Total output rows : {total_out:,}")
print(f"Rows dropped      : {total_in - total_out:,} ({(total_in-total_out)/total_in*100:.2f}%)")
print(f"Temp inversions fixed  : {bad_temp_rows:,}")
print(f"Humidity inversions    : {bad_hum_rows:,} (logged, not fixed)")
print(f"Grid cells 1994 / 2006 : {len(latlon_1994):,} / {len(latlon_2006):,} "
      f"({'MATCH' if latlon_1994==latlon_2006 else 'MISMATCH'})")
print(f"Saved -> {OUTPUT_PATH}")

Batch   1 | in:   250,000 | out:   249,096 | running out:      249,096
Batch  50 | in:   250,000 | out:   249,103 | running out:   12,454,974
Batch 100 | in:   250,000 | out:   249,103 | running out:   24,909,948
Batch 150 | in:   250,000 | out:   249,521 | running out:   37,369,235
Batch 200 | in:   250,000 | out:   249,353 | running out:   49,838,127
Batch 250 | in:   250,000 | out:   249,469 | running out:   62,309,378
Batch 300 | in:   250,000 | out:   249,279 | running out:   74,777,987
Batch 350 | in:   250,000 | out:   249,514 | running out:   87,234,840
Batch 400 | in:   250,000 | out:   248,864 | running out:   99,691,377
Batch 450 | in:   250,000 | out:   249,744 | running out:  112,178,905
Batch 500 | in:   250,000 | out:   249,547 | running out:  124,664,551

Batch cleaning done.
Total input rows  : 127,478,960
Total output rows : 127,137,467
Rows dropped      : 341,493 (0.27%)
Temp inversions fixed  : 2,080
Humidity inversions    : 0 (logged, not fixed)
Grid cells 1994 / 2

## 8. Save Processing Log

In [9]:
missing_rate = (missing_counts / total_in).sort_values(ascending=False)
missing_rate = missing_rate[missing_rate > 0]

log_messages += [
    '=' * 50,
    f"Finished : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
    f"Total input rows  : {total_in:,}",
    f"Total output rows : {total_out:,}",
    f"Rows dropped      : {total_in - total_out:,}",
    f"Temp inversions fixed  : {bad_temp_rows:,}",
    f"Humidity inversions    : {bad_hum_rows:,}",
    f"Grid cells 1994 / 2006 : {len(latlon_1994)} / {len(latlon_2006)}",
    '--- Missing rates (pre-clean) ---',
    missing_rate.to_string(),
    '--- Fill/drop policy ---',
    f"Zero-filled : {FILL_ZERO_COLS}",
    f"Row-dropped  : {DROP_NA_COLS}",
    'PDSI missing values kept as-is (not filled).',
]

log_path = os.path.join(LOG_DIR, LOG_FILE)
with open(log_path, 'w') as f: f.write('\n'.join(log_messages))
print(f"Log saved -> {log_path}")
print('\n'.join(log_messages[-10:]))

Log saved -> E:\zcao\CA_Wildfire\Logs\Feature_Engineering\wind_direction_fill_na_log.txt
Rows dropped      : 341,493
Temp inversions fixed  : 2,080
Humidity inversions    : 0
Grid cells 1994 / 2006 : 13048 / 13048
--- Missing rates (pre-clean) ---
precipitation_amount                         0.573698
LAI                                          0.030399
SWE                                          0.017091
wind_from_direction                          0.002679
min_relative_humidity                        0.001678
wind_speed                                   0.001678
dead_fuel_moisture_1000hr                    0.001678
dead_fuel_moisture_100hr                     0.001678
surface_downwelling_shortwave_flux_in_air    0.001678
max_relative_humidity                        0.001678
specific_humidity                            0.001678
min_air_temperature                          0.001169
max_air_temperature                          0.001169
population_density                           0.000

## 9. Summary

| Phase | Step | Description | Key Result |
|-------|------|-------------|------------|
| A | Sample load | First 200k rows via PyArrow | Quick QA |
| A | Missing rates | Check NaN % per column on sample | Identify fill strategy |
| A | Wind binning | Verify octant bin edges on sample | Distribution confirmed |
| A | Inversion QA | Count bad temp / humidity rows on sample | Pre-flight check |
| A | Grid check | Unique (lat,lon) count — 1994 vs 2006 | Should match |
| B | Batch wind | np.select octant binning per batch | `wind_direction_category` added |
| B | Temp fix | max = min when max < min | Rare edge-case corrected |
| B | Zero fill | `SWE`, `precipitation_amount`, `LAI`, `population_density` | No snow/rain → 0 |
| B | Row drop | Missing core weather variables | Rows removed |
| B | Save | `Daily_Features_Completed.parquet` | Final daily feature table |